# Day 3 — Data Cleaning

This notebook applies conservative, reproducible cleaning to the UCI Online Retail dataset. Business filters are explicit rather than hidden.


In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
from src.data_cleaning import clean_online_retail, quality_summary

dataset = fetch_ucirepo(id=352)
df = dataset.data.features.copy()
df = clean_online_retail(df)
df.head()

## Cleaning decisions

- Parse `InvoiceDate` with invalid values converted to missing.
- Convert quantity, unit price, and customer ID to numeric types.
- Remove exact duplicate records because they contain no additional information.
- Create `revenue = quantity × unit_price` when both fields are available.
- Do **not** silently remove cancellations, non-positive quantities/prices, or missing customer IDs. Those records are profiled first and filtered only when justified by a downstream business question.


In [ ]:
quality_summary(df)

In [ ]:
checks = {}
checks['duplicate_rows'] = int(df.duplicated().sum())
checks['invalid_dates'] = int(df['invoice_date'].isna().sum())
checks['non_positive_quantity'] = int((df['quantity'] <= 0).sum())
checks['non_positive_unit_price'] = int((df['unit_price'] <= 0).sum())
checks['missing_customer_id'] = int(df['customer_id'].isna().sum())
pd.Series(checks, name='count')

## Optional analytical transaction view

For sales and customer-value analyses, a positive-sales view can be created explicitly. This is not written over the cleaned base data, preserving traceability.


In [ ]:
sales_df = df.loc[
    df['quantity'].gt(0) &
    df['unit_price'].gt(0) &
    df['invoice_date'].notna()
].copy()
sales_df.shape

In [ ]:
# Save a processed CSV only when running locally; raw data remains uncommitted.
# sales_df.to_csv('../data/processed/online_retail_sales.csv', index=False)